# 가상환경 확인과 첫 데이터 읽기

> **11회 가상환경과 Colab** 실습 파일입니다.
> 이 노트북과 `이산화탄소_측정기록_실습.xlsx`를 **같은 폴더**에 두고 실행하세요.

## 이 노트북에서 할 일
1. 지금 노트북이 **가상환경(.venv)의 파이썬**으로 돌아가는지 확인
2. 설치한 라이브러리(pandas 등) 확인
3. 05회에 엑셀 매크로로 했던 **CO2 요약표**를 파이썬으로 만들어 보기
4. 같은 노트북을 **Google Colab**에서도 실행해 보기

## 1. 어떤 파이썬으로 실행 중인가?

In [ ]:
import sys

print("파이썬 버전 :", sys.version.split()[0])
print("파이썬 위치 :", sys.executable)
print("가상환경 사용:", sys.prefix != sys.base_prefix)

- 내 PC에서 실행했다면 **파이썬 위치**에 `.venv`가 들어 있어야 합니다. 예: `.../ai-black/.venv/bin/python`
- `가상환경 사용: True` 이면 성공입니다.
- Colab에서 실행하면 `/usr/bin/python3` 같은 Colab 서버의 위치가 나옵니다.

## 2. 라이브러리 확인

In [ ]:
import pandas as pd
import openpyxl
import matplotlib

print("pandas     :", pd.__version__)
print("openpyxl   :", openpyxl.__version__)
print("matplotlib :", matplotlib.__version__)

`ModuleNotFoundError`가 나오면 라이브러리가 설치되지 않은 것입니다.
VS Code 터미널에서 가상환경이 켜져 있는지(`(.venv)` 표시) 확인하고 `pip install -r requirements.txt`를 실행하세요.

## 3. 엑셀 파일 읽기

`이산화탄소_측정기록_실습.xlsx`는 05회에 쓴 **가상 데이터** 파일입니다.

| 시트 | 내용 |
|---|---|
| `요약` | 비어 있는 요약표 |
| `측정1`~`측정3` | 1행: 제목, 2행: 열 이름, 3행부터 측정값 |

먼저 파일 안의 **시트 이름 목록**을 보고, `요약`을 뺀 **측정 시트만** 읽습니다.
`header=1`은 "**두 번째 줄**(0부터 세서 1)을 열 이름으로 써라"라는 뜻입니다. 첫 줄은 제목이기 때문입니다.

In [ ]:
file = "이산화탄소_측정기록_실습.xlsx"

# 1) 파일 안의 시트 이름 목록
all_names = pd.ExcelFile(file).sheet_names
print("파일 안의 시트:", all_names)

# 2) '요약'을 뺀 측정 시트만 고르기
data_names = [name for name in all_names if name != "요약"]

# 3) 측정 시트들을 한 번에 읽기 → {시트이름: 표} 모양의 사전(dict)
sheets = pd.read_excel(file, sheet_name=data_names, header=1)
print("읽은 측정 시트:", list(sheets.keys()))

In [ ]:
# '측정1' 시트의 앞부분 5줄 보기
sheets["측정1"].head()

## 4. 05회의 요약표를 파이썬으로

05회에는 **엑셀 매크로(VBA)** 로 시트마다 평균·최대·최소·판정을 구했습니다.
같은 일을 파이썬으로 하면 아래와 같습니다. 지금은 코드를 다 이해하지 않아도 됩니다. **결과가 05회와 같은지**만 확인하세요.

In [ ]:
limit = 1000
rows = []

for name, table in sheets.items():   # 측정 시트를 하나씩 꺼내서
    co2 = table["CO2 ppm"]
    average = round(co2.mean(), 1)
    rows.append({
        "측정지점": name,
        "평균(ppm)": average,
        "최대(ppm)": co2.max(),
        "최소(ppm)": co2.min(),
        "판정": "부적합" if average >= limit else "적합",
    })

summary = pd.DataFrame(rows)
summary

05회 해답과 비교해 보세요: 측정1 **667.4**, 측정2 **1052.8 (부적합)**, 측정3 **554.9**

## 5. 그래프로 보기 (맛보기)
그래프는 5단계에서 자세히 배웁니다. 한글 글꼴 설정 전이라 그래프 글자는 영어로 적었습니다.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.5))
plt.bar(["Point 1", "Point 2", "Point 3"], summary["평균(ppm)"], color=["#4472C4", "#C00000", "#4472C4"])
plt.axhline(limit, color="red", linestyle="--", label="Standard 1,000 ppm")
plt.ylabel("CO2 average (ppm)")
plt.legend()
plt.tight_layout()
plt.show()

---
## 6. Colab에서도 실행해 보기
1. `https://colab.research.google.com` → **[파일] → [노트북 업로드]** → 이 노트북 선택
2. 왼쪽 **📁 파일** 아이콘 → **[업로드]** → `이산화탄소_측정기록_실습.xlsx` 업로드
3. **[런타임] → [모두 실행]**
4. 1번 셀의 **파이썬 위치**가 내 PC와 어떻게 다른지 비교해 보세요.

> ⚠️ Colab에 올린 파일은 **구글 서버**로 갑니다. 실제 업무 데이터는 Colab에 올리지 말고 **내 PC의 가상환경**에서 처리하세요. (06회 안전 수칙)